# Canonical SU(3) O(u^4) production

Use a standard **CPU** Colab runtime. Run all cells, upload only `ENGINE_Y4_hodge_canonical_o4_production_colab.py` when prompted, and leave the run directory unchanged so an interrupted Stage-3G calculation can resume. The first code cell computes and seals the 189-entry kernel without any historical coefficient. The second cell performs the terminal comparison only after those seals exist.

In [ ]:
from google.colab import files
from pathlib import Path
from fractions import Fraction
import errno, fcntl, gzip, hashlib, json, math, os, subprocess, sys

SCRIPT_NAME = 'ENGINE_Y4_hodge_canonical_o4_production_colab.py'
EXPECTED_SCRIPT_SHA256 = '1970c63a426812bece12b1be1706958fd8ea9ecfbeb3d305875d40ff6f2266b5'
SCRIPT_SNAPSHOT = Path('/content') / f'{SCRIPT_NAME}.{EXPECTED_SCRIPT_SHA256}.verified.py'
RUN_ROOT = Path('/content/Y4_CANONICAL_PRODUCTION')
REFERENCE_PATH = Path('/content/.Y4_HISTORICAL_Q3_TERMINAL_ONLY.json')

if REFERENCE_PATH.exists():
    raise RuntimeError(f'Stale terminal reference exists; remove it before construction: {REFERENCE_PATH}')
uploaded = files.upload()
if SCRIPT_NAME not in uploaded:
    raise RuntimeError(f'Upload exactly {SCRIPT_NAME}')
uploaded_bytes = bytes(uploaded[SCRIPT_NAME])
observed = hashlib.sha256(uploaded_bytes).hexdigest()
if observed != EXPECTED_SCRIPT_SHA256:
    raise RuntimeError(f'Wrong script bytes: {observed} != {EXPECTED_SCRIPT_SHA256}')
if SCRIPT_SNAPSHOT.exists():
    if SCRIPT_SNAPSHOT.read_bytes() != uploaded_bytes:
        raise RuntimeError(f'Hash-named script snapshot collision: {SCRIPT_SNAPSHOT}')
else:
    SCRIPT_SNAPSHOT.write_bytes(uploaded_bytes)
    SCRIPT_SNAPSHOT.chmod(0o444)

def run_hash_bound(script_snapshot, sealed_script_bytes, expected_sha256, arguments):
    def current_hash():
        if not script_snapshot.is_file():
            raise RuntimeError(f'Validated script snapshot is missing: {script_snapshot}')
        return hashlib.sha256(script_snapshot.read_bytes()).hexdigest()
    sealed_hash = hashlib.sha256(sealed_script_bytes).hexdigest()
    if sealed_hash != expected_sha256:
        raise RuntimeError(f'In-memory sealed script bytes are wrong: {sealed_hash}')
    before = current_hash()
    if before != expected_sha256:
        raise RuntimeError(f'Script snapshot changed before subprocess: {before}')
    required_seals = fcntl.F_SEAL_WRITE | fcntl.F_SEAL_SHRINK | fcntl.F_SEAL_GROW | fcntl.F_SEAL_SEAL
    sealed_fd = -1
    try:
        if not hasattr(os, 'memfd_create') or not Path('/proc/self/fd').is_dir():
            raise RuntimeError('Linux sealed memfd execution is required')
        sealed_fd = os.memfd_create('hodge_y4_sealed_runner', os.MFD_ALLOW_SEALING | os.MFD_CLOEXEC)
        remaining = memoryview(sealed_script_bytes)
        while remaining:
            written = os.write(sealed_fd, remaining)
            if written <= 0: raise RuntimeError('Short write to sealed runner memfd')
            remaining = remaining[written:]
        os.fsync(sealed_fd)
        if os.pread(sealed_fd, len(sealed_script_bytes) + 1, 0) != sealed_script_bytes:
            raise RuntimeError('Runner memfd copy mismatch before sealing')
        fcntl.fcntl(sealed_fd, fcntl.F_ADD_SEALS, required_seals)
        if int(fcntl.fcntl(sealed_fd, fcntl.F_GET_SEALS)) != required_seals:
            raise RuntimeError('Runner memfd seals are incomplete')
        try:
            os.pwrite(sealed_fd, sealed_script_bytes[:1] or b'x', 0)
        except OSError as exc:
            if exc.errno != errno.EPERM: raise
        else:
            raise RuntimeError('Runner memfd remained writable after sealing')
        if hashlib.sha256(os.pread(sealed_fd, len(sealed_script_bytes) + 1, 0)).hexdigest() != expected_sha256:
            raise RuntimeError('Sealed runner memfd hash mismatch')
        sealed_fd_path = f'/proc/self/fd/{sealed_fd}'
        child_env = os.environ.copy()
        child_env['HODGE_Y4_SEALED_SOURCE_FD'] = str(sealed_fd)
        subprocess.run(
            [sys.executable, '-u', sealed_fd_path, *arguments],
            check=True,
            pass_fds=(sealed_fd,),
            env=child_env,
        )
        if int(fcntl.fcntl(sealed_fd, fcntl.F_GET_SEALS)) != required_seals or hashlib.sha256(os.pread(sealed_fd, len(sealed_script_bytes) + 1, 0)).hexdigest() != expected_sha256:
            raise RuntimeError('Sealed runner memfd changed during subprocess')
    finally:
        if sealed_fd >= 0: os.close(sealed_fd)
    after = current_hash()
    if after != expected_sha256:
        raise RuntimeError(f'Script snapshot changed during subprocess: {after}')

print('UPLOAD SHA256 PASS:', observed)
run_hash_bound(SCRIPT_SNAPSHOT, uploaded_bytes, EXPECTED_SCRIPT_SHA256, ['--self-test'])

binding_path = RUN_ROOT / 'Y4_CANONICAL_RUN_BINDING.json'
status_path = RUN_ROOT / 'Y4_CANONICAL_RUN_STATUS.json'
ready = False
resume_construction = not binding_path.exists() and not status_path.exists()
if binding_path.is_file() and status_path.is_file():
    binding = json.loads(binding_path.read_text(encoding='utf-8'))
    status = json.loads(status_path.read_text(encoding='utf-8'))
    state = status.get('status')
    if binding.get('runtime_script_sha256') != observed:
        raise RuntimeError('Existing run is bound to different script bytes')
    terminal_retry = state == 'FAIL' and status.get('detail', {}).get('phase') == 'terminal_unblind'
    if state in {'PREUNBLIND_PASS', 'PASS'} or terminal_retry:
        verify_args = ['--root', str(RUN_ROOT), '--verify-sealed-run']
        if state == 'PASS':
            verify_args.append('--require-terminal-pass')
        run_hash_bound(SCRIPT_SNAPSHOT, uploaded_bytes, EXPECTED_SCRIPT_SHA256, verify_args)
        ready = True
    elif state == 'RUNNING' or (state == 'FAIL' and status.get('detail', {}).get('phase') == 'canonical_construction'):
        resume_construction = True
    else:
        raise RuntimeError(f'Existing run is not safely resumable: {status}')
elif binding_path.exists() or status_path.exists():
    raise RuntimeError('Existing run has an incomplete binding/status pair')
if ready:
    print('BOUND SEALED RUN FOUND:', status.get('status'))
else:
    if not resume_construction:
        raise RuntimeError('Refusing to start over inside an unverified run directory')
    run_hash_bound(SCRIPT_SNAPSHOT, uploaded_bytes, EXPECTED_SCRIPT_SHA256, ['--root', str(RUN_ROOT), '--resume'])
    status = json.loads(status_path.read_text(encoding='utf-8'))
if status.get('status') not in {'PREUNBLIND_PASS', 'PASS'} and not (status.get('status') == 'FAIL' and status.get('detail', {}).get('phase') == 'terminal_unblind'):
    raise RuntimeError(f'Construction did not reach a sealed state: {status}')
run_hash_bound(SCRIPT_SNAPSHOT, uploaded_bytes, EXPECTED_SCRIPT_SHA256, ['--root', str(RUN_ROOT), '--verify-sealed-run'] + (['--require-terminal-pass'] if status.get('status') == 'PASS' else []))
print('CONSTRUCTION CELL PASS - PHYSICAL ARTIFACTS RECOMPUTED - HISTORICAL REFERENCE NOT CREATED - NO GPU USED')


In [ ]:
def verify_existing_terminal_pass(run_root, expected_script_sha256, reference_commitment):
    def strict_load(raw, label, compressed=False):
        if compressed:
            try:
                raw = gzip.decompress(raw)
            except Exception as exc:
                raise RuntimeError(f'{label} is not valid gzip') from exc
        def reject_constant(value):
            raise RuntimeError(f'{label} contains non-finite JSON: {value}')
        def unique(pairs):
            result = {}
            for key, value in pairs:
                if key in result:
                    raise RuntimeError(f'{label} contains duplicate key {key!r}')
                result[key] = value
            return result
        try:
            value = json.loads(raw.decode('utf-8'), parse_constant=reject_constant, object_pairs_hook=unique)
        except RuntimeError:
            raise
        except Exception as exc:
            raise RuntimeError(f'{label} is malformed') from exc
        def finite(item):
            if isinstance(item, float) and not math.isfinite(item):
                raise RuntimeError(f'{label} contains a non-finite number')
            if isinstance(item, dict):
                for child in item.values(): finite(child)
            elif isinstance(item, list):
                for child in item: finite(child)
        finite(value)
        return value
    def exact(value, keys, label):
        if not isinstance(value, dict) or set(value) != set(keys):
            raise RuntimeError(f'{label} schema is not exact')
        return value
    def canon(value):
        return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=True, allow_nan=False).encode('utf-8')
    def digest_raw(raw): return hashlib.sha256(raw).hexdigest()
    paths = {
        'status': run_root / 'Y4_CANONICAL_RUN_STATUS.json',
        'pre': run_root / 'Y4_CANONICAL_CERTIFICATE_PREUNBLIND.json',
        'intrinsic': run_root / 'Y4_GAMMA_INTRINSIC_PREUNBLIND.json',
        'final': run_root / 'Y4_CANONICAL_CERTIFICATE_FINAL.json',
        'snapshot': run_root / 'Y4_KERNEL_SEALED_SNAPSHOT.json.gz',
        'stage_i': run_root / 'Y4_STAGE3I' / 'y4_complete_folded_word_weights.json.gz',
        'stage_j_kernel': run_root / 'Y4_STAGE3J' / 'DATA_Y4_full_real_space_h4_kernel.json.gz',
        'stage_j_verdict': run_root / 'Y4_STAGE3J' / 'CERT_Y4_stage3j_verdict.json',
    }
    missing = [name for name, path in paths.items() if not path.is_file()]
    if missing: raise RuntimeError(f'PASS is missing physical artifacts: {missing}')
    raw = {name: path.read_bytes() for name, path in paths.items()}
    sha = {name: digest_raw(data) for name, data in raw.items()}
    status = strict_load(raw['status'], 'status')
    pre = exact(strict_load(raw['pre'], 'pre-unblind certificate'), {'schema','runner_version','status','canonical_variable','runtime','authority','normalization','half_history_launch_policy','construction','preunblind','determinism_scope'}, 'pre-unblind certificate')
    pre_runtime = exact(pre['runtime'], {'script_path','script_sha256','python','platform','package_versions'}, 'pre-unblind runtime')
    pre_state = exact(pre['preunblind'], {'kernel_frozen','historical_reference_loaded','historical_reference_available_to_construction'}, 'pre-unblind state')
    intrinsic = strict_load(raw['intrinsic'], 'intrinsic certificate')
    final = strict_load(raw['final'], 'final certificate')
    stage_i = exact(strict_load(raw['stage_i'], 'Stage-I words', True), {'meta','words'}, 'Stage-I words')
    stage_i_meta = exact(stage_i['meta'], {'version','orbit_file','orbit_sha256','rooted_multiplicity'}, 'Stage-I metadata')
    if stage_i_meta['version'] != '2026-06-13-stage3i-v1' or len(stage_i['words']) != 4221:
        raise RuntimeError('Stage-I corpus is not canonical')
    stage_i_identity = {'schema':'hodge-y4-stage3i-word-weights-physical-v1','metadata':{'version':stage_i_meta['version'],'rooted_multiplicity':stage_i_meta['rooted_multiplicity']},'word_count':len(stage_i['words']),'words':stage_i['words']}
    physical_stage_i = digest_raw(canon(stage_i_identity))
    if physical_stage_i != '5a993a94b5802d0f5decad7e847990cc743b6e3204d318b495ef6e2106f04cc6':
        raise RuntimeError('Stage-I physical identity mismatch')
    kernel = exact(strict_load(raw['snapshot'], 'sealed kernel', True), {'meta','kernel'}, 'sealed kernel')
    kernel_meta = exact(kernel['meta'], {'version','stage3i_input','stage3i_sha256','basis_planes'}, 'kernel metadata')
    basis = [[0,1],[0,2],[1,2]]
    if kernel_meta['version'] != '2026-06-13-stage3j-v1' or kernel_meta['basis_planes'] != basis or kernel_meta['stage3i_sha256'] != sha['stage_i'] or raw['stage_j_kernel'] != raw['snapshot'] or len(kernel['kernel']) != 189:
        raise RuntimeError('sealed kernel is not bound to this run Stage-I/Stage-J chain')
    kernel_identity = {'schema':'hodge-y4-kernel-physical-v2','basis_planes':basis,'stage_i_word_weights_physical_identity_sha256':physical_stage_i,'records':kernel['kernel']}
    physical_kernel = digest_raw(canon(kernel_identity))
    if physical_kernel != '797f03063caf06eba2aaaf6cf6c553a165d7642e4d169e9030fc7096fbe7dd9b':
        raise RuntimeError('kernel physical identity mismatch')
    plane_index = {(0,1):0,(0,2):1,(1,2):2}
    matrix = [[Fraction(0) for _ in range(3)] for _ in range(3)]
    sparse = {}
    for record in kernel['kernel']:
        exact(record, {'input_plane','output_plane','displacement','weight'}, 'kernel record')
        inp, out, dv = tuple(record['input_plane']), tuple(record['output_plane']), tuple(record['displacement'])
        weight = Fraction(record['weight'])
        key = (inp,out,dv)
        if key in sparse or inp not in plane_index or out not in plane_index or len(dv) != 3 or weight == 0:
            raise RuntimeError('kernel record is malformed or duplicated')
        sparse[key] = weight
        matrix[plane_index[out]][plane_index[inp]] += weight
    for (inp,out,dv), weight in sparse.items():
        if sparse.get((out,inp,tuple(-value for value in dv))) != weight:
            raise RuntimeError('kernel is not exactly Hermitian')
    diagonal = [matrix[i][i] for i in range(3)]
    if any(matrix[i][j] != 0 for i in range(3) for j in range(3) if i != j) or len(set(diagonal)) != 1:
        raise RuntimeError('Gamma matrix is not exact scalar 3x3')
    computed_q3 = str(diagonal[0])
    verdict = exact(strict_load(raw['stage_j_verdict'], 'Stage-J verdict'), {'meta','verdict','high_symmetry_corrections','gates','files','scope','passed'}, 'Stage-J verdict')
    if set(verdict['gates']) != {'J0_real_space_kernel','J1_cube_boundary','J2_dispersion_witness','J3_round_trip'} or verdict['gates']['J0_real_space_kernel'].get('stage3i_sha256') != sha['stage_i'] or verdict['gates']['J0_real_space_kernel'].get('nonzero_full_kernel_entries') != 189 or verdict.get('passed') is not True:
        raise RuntimeError('Stage-J J0/raw lineage is not exact')
    construction = pre.get('construction', {})
    identities = construction.get('physical_identity_sha256', {})
    kernel_cert = construction.get('kernel', {})
    order_cert = construction.get('four_insertion_order_certificate', {})
    if pre.get('schema') != 'hodge-y4-canonical-preunblind-v1' or pre.get('runner_version') != '2026-08-19-canonical-o4-production-v1' or pre.get('status') != 'PREUNBLIND_CANONICAL_PASS' or pre_runtime.get('script_sha256') != expected_script_sha256 or pre_state != {'kernel_frozen':True,'historical_reference_loaded':False,'historical_reference_available_to_construction':False} or identities.get('stage_i_word_weights') != physical_stage_i or identities.get('kernel') != physical_kernel or construction.get('artifact_sha256', {}).get('Y4_STAGE3I/y4_complete_folded_word_weights.json.gz') != sha['stage_i'] or kernel_cert.get('source_run_raw_sha256') != sha['stage_j_kernel'] or kernel_cert.get('sealed_snapshot_raw_sha256') != sha['snapshot'] or kernel_cert.get('records') != kernel['kernel'] or order_cert.get('passed') is not True or order_cert.get('insertion_count_per_word') != 4:
        raise RuntimeError('pre-unblind certificate does not match physical artifacts')
    expected_matrix = [[str(value) for value in row] for row in matrix]
    if intrinsic.get('passed') is not True or intrinsic.get('historical_reference_loaded') is not False or intrinsic.get('preunblind_certificate_sha256') != sha['pre'] or intrinsic.get('kernel_sha256') != sha['snapshot'] or intrinsic.get('kernel_physical_identity_sha256') != physical_kernel or intrinsic.get('gamma_matrix') != expected_matrix or intrinsic.get('computed_q3_u4') != computed_q3:
        raise RuntimeError('intrinsic Gamma does not recompute from the snapshot')
    detail = status.get('detail', {})
    if status.get('status') != 'PASS' or status.get('runtime_script_sha256') != expected_script_sha256 or detail.get('phase') != 'terminal_unblind' or detail.get('preunblind_sha256') != sha['pre'] or detail.get('intrinsic_gamma_sha256') != sha['intrinsic'] or detail.get('final_sha256') != sha['final'] or detail.get('historical_exact_equal') is not True or Path(detail.get('final_certificate', '')).resolve() != paths['final'].resolve():
        raise RuntimeError('PASS status is not bound to the recomputed seals')
    historical = final.get('historical_reference', {})
    if final.get('passed') is not True or final.get('exact_equal') is not True or final.get('difference') != '0' or final.get('preunblind_certificate_sha256') != sha['pre'] or final.get('intrinsic_gamma_certificate_sha256') != sha['intrinsic'] or final.get('computed_q3_u4') != computed_q3 or detail.get('computed_q3_u4') != computed_q3:
        raise RuntimeError('final certificate does not match recomputed Gamma')
    historical_envelope = {'schema':historical.get('schema'),'label':historical.get('label'),'canonical_variable':historical.get('canonical_variable'),'value_u4':historical.get('value_u4'),'authority_sha256':historical.get('authority_sha256'),'authority_locator':historical.get('authority_locator'),'sealed_for_terminal_only':historical.get('sealed_for_terminal_only')}
    expected_historical_envelope = {'schema':'hodge-y4-historical-q3-reference-v1','label':'historical SU(3) 189-kernel q3 reference','canonical_variable':'u=beta_H/6=1/g_H^4','value_u4':computed_q3,'authority_sha256':'935a3a5ba680d1373a5842486b10231d83232d8cb3393bbc250351bc51a68c8b','authority_locator':'v24c line 7311; canonical v1.4 master lines 443-458','sealed_for_terminal_only':True}
    if historical_envelope != expected_historical_envelope or digest_raw(canon(historical_envelope)) != reference_commitment or historical.get('reference_envelope_sha256') != reference_commitment or historical.get('loaded_after_kernel_and_gamma_seals') is not True:
        raise RuntimeError('historical reference commitment is not exact')
    return final

status_path = RUN_ROOT / 'Y4_CANONICAL_RUN_STATUS.json'
status = json.loads(status_path.read_text(encoding='utf-8'))
if status.get('status') == 'PASS':
    run_hash_bound(SCRIPT_SNAPSHOT, uploaded_bytes, EXPECTED_SCRIPT_SHA256, ['--root', str(RUN_ROOT), '--verify-sealed-run', '--require-terminal-pass'])
    verify_existing_terminal_pass(RUN_ROOT, EXPECTED_SCRIPT_SHA256, 'a85b5eca065fba3d41278cfddc5d8a3b7d6c1462c29d4c765a7c5ad266d8d329')
    print('TERMINAL HISTORICAL COMPARISON PASS (existing sealed run)')
    print('CANONICAL PRODUCTION PASS - NO GPU USED')
else:
    reference = {
        'schema': 'hodge-y4-historical-q3-reference-v1',
        'label': 'historical SU(3) 189-kernel q3 reference',
        'canonical_variable': 'u=beta_H/6=1/g_H^4',
        'value_u4': '-20721577909065127111/7250590288602460800',
        'authority_sha256': '935a3a5ba680d1373a5842486b10231d83232d8cb3393bbc250351bc51a68c8b',
        'authority_locator': 'v24c line 7311; canonical v1.4 master lines 443-458',
        'sealed_for_terminal_only': True,
    }
    reference_raw = json.dumps(reference, sort_keys=True, separators=(',', ':'), ensure_ascii=True, allow_nan=False).encode('utf-8')
    commitment = hashlib.sha256(reference_raw).hexdigest()
    if commitment != 'a85b5eca065fba3d41278cfddc5d8a3b7d6c1462c29d4c765a7c5ad266d8d329':
        raise RuntimeError(f'Historical envelope commitment mismatch: {commitment}')
    REFERENCE_PATH.write_bytes(reference_raw)
    try:
        run_hash_bound(SCRIPT_SNAPSHOT, uploaded_bytes, EXPECTED_SCRIPT_SHA256, ['--root', str(RUN_ROOT), '--terminal-unblind', '--reference', str(REFERENCE_PATH)])
    finally:
        REFERENCE_PATH.unlink(missing_ok=True)
    status = json.loads(status_path.read_text(encoding='utf-8'))
    if status.get('status') != 'PASS':
        raise RuntimeError(f'Terminal comparison did not pass: {status}')
    run_hash_bound(SCRIPT_SNAPSHOT, uploaded_bytes, EXPECTED_SCRIPT_SHA256, ['--root', str(RUN_ROOT), '--verify-sealed-run', '--require-terminal-pass'])
    verify_existing_terminal_pass(RUN_ROOT, EXPECTED_SCRIPT_SHA256, 'a85b5eca065fba3d41278cfddc5d8a3b7d6c1462c29d4c765a7c5ad266d8d329')
    print('NOTEBOOK TERMINAL PASS - REFERENCE ENVELOPE REMOVED - NO GPU USED')
